In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [2]:
house_dataset = pd.read_csv('housing.csv')

In [3]:
house_dataset

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [4]:
house_dataset['ocean_proximity'].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [5]:
features = house_dataset.drop(columns=['median_house_value'])
features

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY
...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,INLAND


In [6]:
target = house_dataset['median_house_value']
target

0        452600.0
1        358500.0
2        352100.0
3        341300.0
4        342200.0
           ...   
20635     78100.0
20636     77100.0
20637     92300.0
20638     84700.0
20639     89400.0
Name: median_house_value, Length: 20640, dtype: float64

In [7]:
house_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [8]:
x_train, x_test, y_train, y_test = train_test_split(features, target, train_size=0.80, random_state=23)

In [9]:
x_train_num = x_train.drop(columns=['ocean_proximity'])

In [10]:
x_train_cat = x_train[['ocean_proximity']]

In [11]:
num_pipeline = Pipeline(
    [
        ('Imputer',SimpleImputer(strategy='mean')),
        ('Standard',StandardScaler())
    ]
)

In [12]:
cat_pipeline = Pipeline(
    [
        ('Imputercat',SimpleImputer(strategy='most_frequent')),
        ('Onehot',OneHotEncoder())
    ]
)

In [13]:
numName = x_train_num.columns

In [14]:
catName = x_train_cat.columns

In [15]:
finalePipeline = ColumnTransformer(
    [
        ('num',num_pipeline,numName),
        ('cat',cat_pipeline,catName)
    ]
)

In [16]:
final_x_train = finalePipeline.fit_transform(x_train)
final_x_train[0]

array([ 0.65982882, -0.69871042,  1.30880591, -0.30718322, -0.23914893,
       -0.16239887, -0.18469775,  0.08459702,  1.        ,  0.        ,
        0.        ,  0.        ,  0.        ])

In [17]:
HousingModel = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42, max_features='sqrt', bootstrap=False, n_jobs=-1)

In [18]:
HousingModel.fit(final_x_train, y_train)

RandomForestRegressor(bootstrap=False, max_depth=20, max_features='sqrt',
                      n_jobs=-1, random_state=42)

In [19]:
final_predict = HousingModel.predict(final_x_train)

In [20]:
r2_score(y_train,final_predict)

0.998242617767242

In [21]:
model = Pipeline(
    [
        ('processor',finalePipeline),
        ('regressor',HousingModel)
    ]
)

In [23]:
predict_test = model.predict(x_test)

In [24]:
r2_score(y_test,predict_test)

0.8286369799995572

# Save Model

In [26]:
joblib.dump(model, 'FinalModel.pkl')

['FinalModel.pkl']